## Set Up

In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

In [43]:
url_1 = 'https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/data/processed/sf_business_demographics_nhood_naics.parquet'
combined = pd.read_parquet(url_1)

In [44]:
url_2 = 'https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/data/processed/sf_businesses_nhood_naics.parquet'
neighborhood_table = pd.read_parquet(url_2)

In [45]:
# compute ratio per neighborhood per period from neighborhood_table

neighborhood_table = neighborhood_table[neighborhood_table['biz_stock'] >= 50]

covid_period   = neighborhood_table[neighborhood_table['year'].between(2020, 2021)]
recovery_period = neighborhood_table[neighborhood_table['year'].between(2022, 2025)]


def period_ratio(period_df):
    agg = period_df.groupby('nhood')[['opened', 'closed']].sum()
    agg['ratio'] = agg['opened'] / agg['closed'].replace(0, float('nan'))
    return agg['ratio'].reset_index()

covid_ratio    = period_ratio(covid_period).rename(columns={'ratio': 'covid_ratio'})
recovery_ratio = period_ratio(recovery_period).rename(columns={'ratio': 'recovery_ratio'})

combined_ratio = covid_ratio.merge(recovery_ratio, on=['nhood'], how='outer')
combined_ratio = combined_ratio.rename(columns={'nhood': 'neighborhood'})

# merge demographics
demo_cols = combined[['neighborhood', 'median_income', 'pct_poc', 'pct_white', 
                       'pct_black', 'pct_asian', 'pct_latina_o', 'pct_other']].drop_duplicates('neighborhood')
combined_ratio = combined_ratio.merge(demo_cols, on='neighborhood', how='left')

# resilience = recovery_ratio / covid_ratio
combined_ratio['resilience'] = combined_ratio['recovery_ratio'] / combined_ratio['covid_ratio']


## San Francisco Business Openings vs Closings (2016–2025)

## COVID Impact vs Recovery by Neighborhood

In [9]:
def get_plot_df(naics_group):
    if naics_group == 'All':
        return combined_ratio.groupby('neighborhood')[['covid_ratio', 'recovery_ratio']].mean().reset_index()
    return combined_ratio[combined_ratio['naics_group'] == naics_group][['neighborhood', 'covid_ratio', 'recovery_ratio']]

def make_covid_fig(naics_group, highlight):
    plot_df = get_plot_df(naics_group)

    x_mean, y_mean = plot_df['covid_ratio'].mean(), plot_df['recovery_ratio'].mean()
    x_min, x_max = plot_df['covid_ratio'].min(), plot_df['covid_ratio'].max()
    y_min, y_max = plot_df['recovery_ratio'].min(), plot_df['recovery_ratio'].max()
    x_pad = (x_max - x_min) * 0.08
    y_pad = (y_max - y_min) * 0.08

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_df['covid_ratio'],
        y=plot_df['recovery_ratio'],
        mode='markers',
        text=plot_df['neighborhood'],
        hovertemplate='<b>%{text}</b><br>COVID ratio: %{x:.2f}<br>Recovery ratio: %{y:.2f}<extra></extra>',
        marker=dict(
            color=['steelblue' if n == highlight else 'lightgray' for n in plot_df['neighborhood']],
            size=[14 if n == highlight else 10 for n in plot_df['neighborhood']]
        ),
    ))

    fig.add_shape(type='line', x0=x_mean, x1=x_mean, y0=y_min, y1=y_max, line=dict(dash='dash', color='black'))
    fig.add_shape(type='line', x0=x_min, x1=x_max, y0=y_mean, y1=y_mean, line=dict(dash='dash', color='black'))
    fig.add_shape(type='line', x0=x_min, x1=x_max, y0=1, y1=1, line=dict(dash='dot', color='gray'))
    fig.add_shape(type='line', x0=1, x1=1, y0=y_min, y1=y_max, line=dict(dash='dot', color='gray'))

    for x, y, text in [
        (x_max - x_pad, y_max - y_pad, 'High COVID ratio<br>High recovery ratio'),
        (x_min + x_pad, y_max - y_pad, 'Low COVID ratio<br>High recovery ratio'),
        (x_min + x_pad, y_min + y_pad, 'Low COVID ratio<br>Low recovery ratio'),
        (x_max - x_pad, y_min + y_pad, 'High COVID ratio<br>Low recovery ratio'),
    ]:
        fig.add_annotation(x=x, y=y, text=text, showarrow=False,
                           font=dict(size=10, color='gray'), bgcolor='rgba(255,255,255,0.6)')

    fig.update_layout(
        title=f'COVID Impact vs Recovery Ratio — {naics_group}',
        xaxis_title='Open/Close Ratio (2020–2021)',
        yaxis_title='Open/Close Ratio (2022–2025)',
        height=500
    )
    return fig

sector_dd = widgets.Dropdown(options=['All'] + sorted(combined_ratio['naics_group'].unique().tolist()), value='All', description='Sector:')
nhood_dd  = widgets.Dropdown(options=['None'] + sorted(combined_ratio['neighborhood'].unique().tolist()), value='None', description='Highlight:')
out = widgets.Output()

def on_change(_):
    with out:
        out.clear_output(wait=True)
        display(make_covid_fig(sector_dd.value, nhood_dd.value))

sector_dd.observe(on_change, names='value')
nhood_dd.observe(on_change, names='value')

with out:
    display(make_covid_fig('All', 'None'))

display(widgets.HBox([sector_dd, nhood_dd]), out)

Output()

## Race/Ethnicity and Median Income by Neighborhood

In [10]:
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import Dropdown, Output, VBox, HBox, SelectMultiple
from IPython.display import display

out = Output()

def plot_neighborhoods(neighborhoods):
    if not neighborhoods:
        return

    race_cols   = ["pct_white", "pct_black", "pct_asian", "pct_latina_o", "pct_other"]
    race_labels = ["White", "Black", "Asian/PI", "Latino", "Other"]
    colors = ["#378ADD", "#E87040", "#4CAF50", "#9C27B0"]

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [3, 1]})

    n = len(neighborhoods)
    bar_width = 0.8 / n

    for i, neighborhood in enumerate(neighborhoods):
        row = combined[combined['neighborhood'] == neighborhood].iloc[0]
        values = [row[c] * 100 for c in race_cols]
        x = [j + i * bar_width for j in range(len(race_cols))]
        ax.bar(x, values, width=bar_width, color=colors[i], label=neighborhood)

    ax.set_xticks([j + bar_width * (n - 1) / 2 for j in range(len(race_cols))])
    ax.set_xticklabels(race_labels)
    ax.set_ylabel("% of population")
    ax.set_ylim(0, 100)
    ax.legend()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax2.axis("off")
    y_pos = 0.9
    for i, neighborhood in enumerate(neighborhoods):
        row = combined[combined['neighborhood'] == neighborhood].iloc[0]
        income = row["median_income"]
        income_str = f"${income:,.0f}" if pd.notna(income) else "No data"
        ax2.text(0.5, y_pos, neighborhood, ha="center", fontsize=9, color="gray", transform=ax2.transAxes)
        ax2.text(0.5, y_pos - 0.08, income_str, ha="center", fontsize=14,
                 fontweight="bold", color=colors[i], transform=ax2.transAxes)
        y_pos -= 0.25

    fig.suptitle("Race/Ethnicity by Neighborhood", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

selector = SelectMultiple(
    options=sorted(combined['neighborhood'].unique().tolist()),
    value=['Sunset/Parkside', 'Bayview Hunters Point'],
    rows=8,
    description='Neighborhoods:'
)

def on_change(change):
    if len(change['new']) > 4:
        selector.value = change['new'][:4]
        return
    with out:
        out.clear_output(wait=True)
        plot_neighborhoods(list(change['new']))

selector.observe(on_change, names='value')



with out:
    plot_neighborhoods(list(selector.value))

display(HBox([selector, out]))

## Business Resilience by Neighborhood Demographics

In [18]:
def make_resilience_fig(naics_group, metric):
    if naics_group == 'All':
        plot_df = combined_ratio.groupby('neighborhood')[['median_income', 'pct_poc', 'recovery_ratio', 'covid_ratio']].mean().reset_index()
    else:
        plot_df = combined_ratio[combined_ratio['naics_group'] == naics_group].copy()

    metric_label = 'Recovery Ratio (2022–2025)' if metric == 'recovery_ratio' else 'COVID Ratio (2020–2021)'
    title_label  = 'Recovery' if metric == 'recovery_ratio' else 'COVID Impact'

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f'{title_label} vs Median Income', f'{title_label} vs % People of Color'),
        horizontal_spacing=0.12
    )

    fig.add_trace(go.Scatter(
        x=plot_df['median_income'], y=plot_df[metric],
        mode='markers+text', text=plot_df['neighborhood'],
        textposition='top center', textfont=dict(size=7, color='gray'),
        hovertemplate=f'<b>%{{text}}</b><br>{metric_label}: %{{y:.2f}}<br>Income: $%{{x:,.0f}}<extra></extra>',
        marker=dict(color=plot_df['median_income'], colorscale='RdBu', size=10,
                    showscale=True, colorbar=dict(title='Median Income', x=0.44),
                    line=dict(width=1, color='white')),
        showlegend=False
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=plot_df['pct_poc'], y=plot_df[metric],
        mode='markers+text', text=plot_df['neighborhood'],
        textposition='top center', textfont=dict(size=7, color='gray'),
        hovertemplate=f'<b>%{{text}}</b><br>{metric_label}: %{{y:.2f}}<br>% POC: %{{x:.1%}}<extra></extra>',
        marker=dict(color=plot_df['pct_poc'], colorscale='RdBu_r', size=10,
                    showscale=True, colorbar=dict(title='% POC', x=1.02),
                    line=dict(width=1, color='white')),
        showlegend=False
    ), row=1, col=2)

    for col in [1, 2]:
        fig.add_shape(type='line', x0=0, x1=1, y0=1, y1=1,
                      xref=f"x{'' if col==1 else col} domain",
                      yref=f"y{'' if col==1 else col}",
                      line=dict(dash='dash', color='black', width=1), row=1, col=col)

    fig.update_xaxes(title_text='Median Income', tickprefix='$', tickformat=',', col=1)
    fig.update_xaxes(title_text='% People of Color', tickformat='.0%', col=2)
    fig.update_yaxes(title_text=metric_label, col=1)
    fig.update_layout(
        title=f'Business {title_label} by Demographics — {naics_group}',
        width=None,
        height=560,
        plot_bgcolor='white'
    )
    return fig

sector_dd2 = widgets.Dropdown(options=['All'] + sorted(combined_ratio['naics_group'].unique().tolist()), value='All', description='Sector:')
metric_dd2  = widgets.Dropdown(options=[('Recovery Ratio', 'recovery_ratio'), ('COVID Ratio', 'covid_ratio')], value='recovery_ratio', description='Metric:')
out2 = widgets.Output()

def on_change2(_):
    with out2:
        out2.clear_output(wait=True)
        display(make_resilience_fig(sector_dd2.value, metric_dd2.value))

sector_dd2.observe(on_change2, names='value')
metric_dd2.observe(on_change2, names='value')

with out2:
    display(make_resilience_fig('All', 'recovery_ratio'))

display(widgets.HBox([sector_dd2, metric_dd2]), out2)

Output()

## Pre vs Post COVID Business Trajectory by Demographic Quartile

In [16]:
def make_trajectory_fig(naics_group):
    if naics_group == 'All':
        nt = neighborhood_table.copy()
    else:
        nt = neighborhood_table[neighborhood_table['naics_group'] == naics_group].copy()

    nt = nt.groupby(['nhood', 'year'])[['opened', 'closed']].sum().reset_index()
    nt['open_close_ratio'] = nt['opened'] / nt['closed'].replace(0, float('nan'))
    nt = nt.merge(demo_cols, left_on='nhood', right_on='neighborhood', how='left')

    nhood_demo = nt[['nhood', 'median_income', 'pct_poc']].drop_duplicates('nhood')
    nhood_demo = nhood_demo.copy()
    nhood_demo['income_quartile'] = pd.qcut(nhood_demo['median_income'], q=4,
        labels=['Q1 (lowest income)', 'Q2', 'Q3', 'Q4 (highest income)'])
    nhood_demo['poc_quartile'] = pd.qcut(nhood_demo['pct_poc'], q=4,
        labels=['Q1 (least POC)', 'Q2', 'Q3', 'Q4 (most POC)'])

    nt = nt.merge(nhood_demo[['nhood', 'income_quartile', 'poc_quartile']], on='nhood', how='left')

    years = list(range(2019, 2026))
    nt = nt[nt['year'].isin(years)]

    income_trends = nt.groupby(['income_quartile', 'year'], observed=True)['open_close_ratio'].mean().unstack('year')
    poc_trends    = nt.groupby(['poc_quartile',    'year'], observed=True)['open_close_ratio'].mean().unstack('year')

    income_palette = ['#d6604d', '#f4a582', '#92c5de', '#2166ac']
    poc_palette    = ['#2166ac', '#92c5de', '#f4a582', '#d6604d']

    fig = make_subplots(rows=1, cols=2,
        subplot_titles=('Avg Open/Close Ratio by Income Quartile',
                        'Avg Open/Close Ratio by % POC Quartile'),
        horizontal_spacing=0.12)

    for i, (q_label, row) in enumerate(income_trends.iterrows()):
        fig.add_trace(go.Scatter(
            x=years, y=row.values, mode='lines+markers',
            name=str(q_label), legendgroup=f'income_{q_label}',
            line=dict(color=income_palette[i], width=2.5), marker=dict(size=7),
            hovertemplate=f'<b>{q_label}</b><br>Year: %{{x}}<br>Ratio: %{{y:.2f}}<extra></extra>'
        ), row=1, col=1)

    for i, (q_label, row) in enumerate(poc_trends.iterrows()):
        fig.add_trace(go.Scatter(
            x=years, y=row.values, mode='lines+markers',
            name=str(q_label), legendgroup=f'poc_{q_label}',
            line=dict(color=poc_palette[i], width=2.5), marker=dict(size=7),
            hovertemplate=f'<b>{q_label}</b><br>Year: %{{x}}<br>Ratio: %{{y:.2f}}<extra></extra>'
        ), row=1, col=2)

    for col in [1, 2]:
        fig.add_shape(type='line', x0=2020, x1=2020, y0=0, y1=3,
                      line=dict(dash='dash', color='black', width=1), row=1, col=col)
        fig.add_annotation(x=2020, y=2.8, text='COVID', showarrow=False,
                           font=dict(size=9, color='black'),
                           xref=f"x{'' if col==1 else col}",
                           yref=f"y{'' if col==1 else col}")
        fig.add_shape(type='line', x0=2019, x1=2025, y0=1, y1=1,
                      line=dict(dash='dot', color='gray', width=1), row=1, col=col)

    fig.update_xaxes(tickvals=years, ticktext=[str(y) for y in years])
    fig.update_yaxes(title_text='Avg Open/Close Ratio', col=1)
    fig.update_layout(title=f'Pre vs Post COVID Ratio Trajectory — {naics_group}',
                      width=1200, height=520, plot_bgcolor='white',
                      legend=dict(orientation='v', x=1.02, y=0.5))
    return fig

sector_dd3 = widgets.Dropdown(
    options=['All'] + sorted(neighborhood_table['naics_group'].unique().tolist()),
    value='All', description='Sector:')
out3 = widgets.Output()

def on_change3(_):
    with out3:
        out3.clear_output(wait=True)
        display(make_trajectory_fig(sector_dd3.value))

sector_dd3.observe(on_change3, names='value')

with out3:
    display(make_trajectory_fig('All'))

display(widgets.VBox([sector_dd3, out3]))